# Clustering & Dimensionality Reduction

Unsupervised learning: finding structure without labels.
Key challenge: evaluating quality without ground truth.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. KMeans — choosing k with the silhouette score

The elbow method is subjective. The silhouette score is objective:
measures how similar each point is to its own cluster vs. the nearest other cluster.
Range: -1 (wrong cluster) to +1 (tight, well-separated cluster).

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

rng = np.random.default_rng(42)
X_raw, y_true = make_blobs(n_samples=600, centers=4, cluster_std=1.1,
                            random_state=42)
X = StandardScaler().fit_transform(X_raw)

# Sweep k
k_range = range(2, 10)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(k, random_state=42, n_init=10).fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

best_k = k_range[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Elbow plot
ax = axes[0]
ax.plot(list(k_range), inertias, "o-", color=C[0], lw=2)
ax.set(xlabel="k", ylabel="Inertia (within-cluster SS)",
       title="Elbow plot
(elbow is subjective - hard to read)")

# Silhouette plot
ax = axes[1]
ax.plot(list(k_range), sil_scores, "o-", color=C[2], lw=2)
ax.axvline(best_k, color=C[1], lw=2, linestyle="--", label=f"Best k={best_k}")
ax.set(xlabel="k", ylabel="Silhouette score",
       title="Silhouette score vs k
(peak = optimal k, objective)")
ax.legend()

# Cluster scatter at best k
ax = axes[2]
km_best = KMeans(best_k, random_state=42, n_init=10).fit(X)
for c in range(best_k):
    mask = km_best.labels_ == c
    ax.scatter(X[mask,0], X[mask,1], s=20, alpha=0.7, color=C[c], label=f"Cluster {c}")
centers = km_best.cluster_centers_
ax.scatter(centers[:,0], centers[:,1], marker="X", s=180, color="black", zorder=5, label="Centroids")
ax.set(xlabel="PC1", ylabel="PC2", title=f"KMeans k={best_k}
(silhouette={sil_scores[best_k-2]:.3f})")
ax.legend(fontsize=8, markerscale=1.5)

plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {best_k}  (true k=4)")
print(f"Best silhouette:      {max(sil_scores):.3f}")

## 2. PCA — variance explained and biplots

PCA finds the directions of maximum variance. The scree plot shows
how many components capture "most" of the variance.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

X_wine, y_wine = load_wine(return_X_y=True)
feature_names   = load_wine().feature_names
X_sc            = StandardScaler().fit_transform(X_wine)
pca             = PCA().fit(X_sc)
X_pca           = pca.transform(X_sc)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scree plot
ax = axes[0]
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
n_comp = np.argmax(cumvar >= 80) + 1
ax.bar(range(1, len(cumvar)+1), pca.explained_variance_ratio_*100,
       color=C[0], alpha=0.8, label="Individual")
ax.plot(range(1, len(cumvar)+1), cumvar, "o-", color=C[1], lw=2, label="Cumulative")
ax.axhline(80, color="grey", lw=1.5, linestyle="--", label="80% threshold")
ax.axvline(n_comp, color=C[2], lw=2, linestyle="--")
ax.set(xlabel="Principal component", ylabel="Variance explained (%)",
       title=f"Scree plot
({n_comp} PCs explain 80% of variance)")
ax.legend(fontsize=9)

# Scatter PC1 vs PC2
ax = axes[1]
wine_classes = load_wine().target_names
for cls, col in zip([0,1,2], C):
    mask = y_wine == cls
    ax.scatter(X_pca[mask,0], X_pca[mask,1], s=30, alpha=0.8, color=col,
               label=wine_classes[cls])
ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
       title="PC1 vs PC2
(wine varieties well-separated)")
ax.legend()

# Biplot (loadings)
ax = axes[2]
loadings = pca.components_[:2].T
scale    = 3
ax.scatter(X_pca[:,0], X_pca[:,1], s=10, alpha=0.2, color="grey")
for i, (lx, ly) in enumerate(loadings):
    ax.annotate("", xy=(lx*scale, ly*scale), xytext=(0,0),
                arrowprops=dict(arrowstyle="->", color=C[1], lw=1.5))
    ax.text(lx*scale*1.1, ly*scale*1.1, feature_names[i],
            fontsize=7, ha="center", color=C[1])
ax.set(xlabel="PC1", ylabel="PC2",
       title="Biplot: feature loadings
(arrow direction = feature contribution)")
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
plt.tight_layout()
plt.show()
print(f"Top 2 features driving PC1: {[feature_names[i] for i in np.argsort(np.abs(pca.components_[0]))[-2:][::-1]]}")

## 3. DBSCAN — density-based clustering (handles noise)

KMeans requires specifying k and assumes spherical clusters.
DBSCAN discovers arbitrary-shaped clusters and labels outliers as noise.

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import silhouette_score

datasets = [
    ("Two moons",   *make_moons(500,  noise=0.08, random_state=42)),
    ("Two circles", *make_circles(500, noise=0.05, factor=0.5, random_state=42)),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, (name, X_d, y_d) in enumerate(datasets):
    # True labels
    for cls in np.unique(y_d):
        axes[row,0].scatter(X_d[y_d==cls,0], X_d[y_d==cls,1], s=12, alpha=0.7, color=C[cls])
    axes[row,0].set(title=f"{name}
Ground truth", xticks=[], yticks=[])

    # KMeans
    km_labels = KMeans(2, random_state=42, n_init=10).fit_predict(X_d)
    for cls in np.unique(km_labels):
        axes[row,1].scatter(X_d[km_labels==cls,0], X_d[km_labels==cls,1],
                            s=12, alpha=0.7, color=C[cls])
    sil_km = silhouette_score(X_d, km_labels)
    axes[row,1].set(title=f"KMeans k=2
sil={sil_km:.3f} (fails on non-convex)", xticks=[], yticks=[])

    # DBSCAN
    db = DBSCAN(eps=0.15, min_samples=5).fit(X_d)
    db_labels = db.labels_
    noise_mask = db_labels == -1
    for cls in np.unique(db_labels[~noise_mask]):
        axes[row,2].scatter(X_d[db_labels==cls,0], X_d[db_labels==cls,1],
                            s=12, alpha=0.7, color=C[cls])
    axes[row,2].scatter(X_d[noise_mask,0], X_d[noise_mask,1],
                        s=12, alpha=0.5, color="grey", marker="x", label="Noise")
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise    = noise_mask.sum()
    axes[row,2].set(title=f"DBSCAN
{n_clusters} clusters, {n_noise} noise pts", xticks=[], yticks=[])

cols = ["Ground truth", "KMeans k=2", "DBSCAN"]
for ax, col in zip(axes[0], cols):
    ax.set_title(f"{col}
{ax.get_title().split(chr(10))[-1]}", fontsize=10)

fig.suptitle("KMeans fails on non-convex clusters; DBSCAN handles them", fontsize=12)
plt.tight_layout()
plt.show()